# ViBioMIR S05 — semantic chunking only
Select a CUDA GPU runtime (T4/L4/A100 or another available CUDA GPU). CPU preparation, fixed/structural chunking and validation are documented in `docs/CHUNKING_STAGE5.md`. TPU is not implemented.
Upload the manually created ZIP to Google Drive. Use the same prepared pilot for all three strategies. No cell has been executed by the assistant.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BUNDLE = Path('/content/drive/MyDrive/ViBioMIR/stage5_colab_pilot.zip')  # edit
BACKUP_DIR = Path('/content/drive/MyDrive/ViBioMIR/stage5_semantic_pilot')  # unique per input/config
RUN_DIR = Path('/content/stage5_semantic_pilot')  # local locks and atomic commits
PROJECT = Path('/content/vibiomir_stage5')
ENCODER_BATCH_SIZE = 8  # engineering starting point for T4; reduce to 4 if necessary
assert BUNDLE.is_file(), f'Upload the bundle first: {BUNDLE}'
import zipfile
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as archive:
    for info in archive.infolist():
        target = (PROJECT / info.filename).resolve()
        assert PROJECT.resolve() in target.parents, 'Unsafe bundle path'
    archive.extractall(PROJECT)
import os
os.chdir(PROJECT)
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
print('Project:', PROJECT, '\nLocal output:', RUN_DIR, '\nDrive backup:', BACKUP_DIR)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-stage5-colab.txt'], check=True)
print('If Colab asks for a runtime restart, restart and rerun the mount/configuration cell before proceeding.')

In [ ]:
import torch, json
assert torch.cuda.is_available(), 'Select a GPU runtime first; TPU and CPU are not supported by this runner.'
print(torch.__version__, torch.cuda.get_device_name(0))
from vibiomir.chunk_models import load_tokenizer, SentenceEncoder
from vibiomir.chunk_pipeline import build_chunks, validate_chunks
from vibiomir.chunk_colab import restore_checkpoint, save_checkpoint
restore_checkpoint(BACKUP_DIR, RUN_DIR)
if (RUN_DIR / 'manifest.json').exists():
    save_checkpoint(RUN_DIR, BACKUP_DIR)  # finish any copy interrupted in this runtime
from vibiomir.chunking import settings
tokenizer, tokenizer_signature = load_tokenizer(PROJECT / 'tokenizer')
config = settings(json.loads((PROJECT / 'configs/chunking_stage5.json').read_text()))
encoder = SentenceEncoder(tokenizer, batch_size=ENCODER_BATCH_SIZE)  # downloads pinned public weights once
print(json.dumps(encoder.registry, indent=2))

Computation and locks stay on local Colab disk. Each committed batch is copied to Drive and checksummed before the next batch starts. After a disconnection, rerun preceding cells to restore and resume. Do not run two sessions against the same BACKUP_DIR. A batch interrupted during its Drive copy may need recomputation. Do not bypass checksum failures. GPU type/batch size may change across resumes; execution metadata records this. Reproducibility across GPUs is numerical, not bitwise.

In [ ]:
result = build_chunks(PROJECT / 'inputs', RUN_DIR, tokenizer, tokenizer_signature,
    strategy='semantic', config=config, encoder=encoder, encoder_registry=encoder.registry,
    resume=(RUN_DIR / 'manifest.json').is_file(),
    on_batch=lambda batch: save_checkpoint(RUN_DIR, BACKUP_DIR, batch))
save_checkpoint(RUN_DIR, BACKUP_DIR)
print(json.dumps(result, indent=2))

In [ ]:
report = validate_chunks(PROJECT / 'inputs', RUN_DIR, tokenizer, tokenizer_signature)
print(json.dumps(report, indent=2))
save_checkpoint(RUN_DIR, BACKUP_DIR)
# Download BACKUP_DIR to inspect locally.
print('Keep manifest.json, VALIDATION.json and every committed batch-* directory:', BACKUP_DIR)